# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Retrieve and print metadata
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Published: {metadata.datePublished} | Version: {metadata.version}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all RecordSet objects and fields by @id
print("Available Record Sets and their Fields/Columns:\n")
record_sets = []
for rs in dataset.metadata.record_sets:
    print(f"RecordSet: {rs['@id']}, name: {rs.get('name', 'N/A')}")
    record_sets.append(rs['@id'])
    if 'fields' in rs:
        for fld in rs['fields']:
            print(f"  Field: {fld['@id']} - name: {fld.get('name', fld['@id'])} (dataType: {fld.get('dataType', 'N/A')})")
    if 'columns' in rs:
        for col in rs['columns']:
            print(f"  Column: {col['@id']} - name: {col.get('name', col['@id'])} (dataType: {col.get('dataType', 'N/A')})")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# If you know the main data table's record set @id, you can focus on it. 
# Otherwise, extract ALL recordsets.

dataframes = {}

print("Loading records as DataFrames for all record sets ...\n")
for record_set in record_sets:
    # Each record set yields a generator of record dicts
    records = list(dataset.records(record_set=record_set))
    if records:
        dataframes[record_set] = pd.DataFrame(records)
        print(f"Loaded RecordSet [{record_set}]: {len(records)} records, columns: {dataframes[record_set].columns.tolist()}")
    else:
        print(f"RecordSet [{record_set}] yielded no records.")

# Let's pick the first (main) record set for demonstration
main_record_set = record_sets[0] if record_sets else None
if main_record_set and main_record_set in dataframes:
    print(f"\nPreview of '{main_record_set}':")
    display(dataframes[main_record_set].head())
else:
    print("No records found in any record set.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filter records, normalize numeric fields, group and summarize.

In [ ]:
# Pick one numeric field (by @id) for EDA, replaced appropriately below
# (You'll need to check the Data Overview output or columns printed for valid field @ids)

# Example: Suppose 'gad7_total_score' is @id of the GAD-7 total score field
main_df = dataframes[main_record_set]

# Guess possible numeric fields
numeric_fields = [col for col in main_df.columns if any(sub in col.lower() for sub in ['score', 'age', 'phq', 'psq']) and pd.api.types.is_numeric_dtype(main_df[col])]
print(f"Numeric fields detected: {numeric_fields}")

if numeric_fields:
    numeric_field = numeric_fields[0]  # Use first found
    print(f"Using numeric field for EDA: '{numeric_field}'")

    # Filter (example: scores greater than a threshold)
    threshold = 10
    filtered_df = main_df[main_df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold}: {len(filtered_df)} records")
    display(filtered_df.head())

    # Normalize the field
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try grouping by a categorical field
    candidate_group_fields = [col for col in main_df.columns if col != numeric_field and pd.api.types.is_categorical_dtype(main_df[col]) or main_df[col].dtype == object]
    if candidate_group_fields:
        group_field = candidate_group_fields[0]
        print(f"Grouping by field: {group_field}")
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame(name=f"mean_{numeric_field}")
        display(grouped_df.head())
    else:
        print("No suitable categorical group field detected.")
else:
    print("No numeric field detected for EDA.")

## 5. Visualization
Visualize data distributions or field relationships.
(We use `matplotlib` - install if missing.)

In [ ]:
import matplotlib.pyplot as plt

if numeric_fields:
    fig, ax = plt.subplots(figsize=(8,5))
    main_df[numeric_field].hist(ax=ax, bins=15, color='skyblue')
    ax.set_title(f"Distribution of {numeric_field}")
    ax.set_xlabel(numeric_field)
    ax.set_ylabel("Frequency")
    plt.show()

    # If grouped, show bar chart
    if 'group_field' in locals():
        grouped_df.plot(kind='bar', legend=False, alpha=0.7)
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.ylabel(f"Mean {numeric_field}")
        plt.xlabel(group_field)
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion
This notebook demonstrated use of `mlcroissant` to load and explore a Croissant-standards-compliant tabular dataset with record set and field `@id` references.

- **Dataset:** Metadata and records were loaded from the Croissant schema URL.
- **Record Sets / Fields:** All entities were referred to by canonical `@id`.
- **EDA:** We filtered, normalized, and grouped a numeric mental health score field (e.g. GAD-7/PHQ-9/PSQ).
- **Visualization:** The field's distribution and group means were visualized.

Further work could include more detailed field analyses, handling missing data, or exploring relationships between multiple assessments (e.g., GAD-7 vs PHQ-9).